In [1]:
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
import numpy as np
import os
import cv2


class Config:
    # NOTE: These paths match the original notebook configuration.
    # Modify these paths to match your environment.
    BASE_INPUT = '/mgpfs/home/asusanto/_scratch/mobilnet-rice-leaf/dataset/Rice_Leaf_AUG/Rice_Leaf_AUG'
    WORK_DIR = '/mgpfs/home/asusanto/_scratch/mobilnet-rice-leaf/work'
    OUTPUT_DIR = '/workspaces/mobilnet-rice-leaf/work/results'
    MODELS_DIR = '/workspaces/mobilnet-rice-leaf/work/models'
    DATASET_DIR = '/workspaces/mobilnet-rice-leaf/work/dataset'
    
    # Target classes for filtering (4 out of 10 classes)
    TARGET_CLASSES = ['bacterial_leaf_blight', 'brown_spot', 'leaf_blast', 'healthy_rice_leaf']
    
    BATCH_SIZE = 16
    IMG_SIZE_MOBILE = 224
    
    # Model architecture
    DROPOUT_RATE = 0.5
    DENSE_UNITS = 256
    NUM_CLASSES = 4  # Updated for 4 classes
    
    SEED = 42


config = Config()


2026-03-01 13:53:31.294957: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-01 13:53:50.403960: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-01 13:54:03.100213: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [3]:
def get_base_model(model):
    """
    Return (base_model, base_input) where base_model is the pretrained backbone.
    Works for: model = Sequential([base_model, GAP, ...])
    """
    if hasattr(model.layers[0], "layers"):
        return model.layers[0], model.layers[0].input
    else:
        return model, model.input

def get_last_conv_name(base_model):
    """
    Find the last Conv2D layer inside a base model.
    """
    convs = [l for l in base_model.layers if isinstance(l, tf.keras.layers.Conv2D)]
    if not convs:
        raise ValueError("No Conv2D layer found in base_model.")
    return convs[-1].name

def make_gradcam_heatmap(img_array, model, last_conv_layer_name=None, pred_index=None):
    """
    img_array: (1, H, W, 3) preprocessed
    model: full model (backbone + head)
    """
    base_model, base_input = get_base_model(model)

    # auto-detect last conv layer if not given
    if last_conv_layer_name is None:
        last_conv_layer_name = get_last_conv_name(base_model)
        print(f"[GradCAM] Using last conv layer: {last_conv_layer_name}")

    # submodel: input -> last conv activation
    last_conv_layer = base_model.get_layer(last_conv_layer_name)
    last_conv_layer_model = tf.keras.models.Model(base_input, last_conv_layer.output)

    # classifier head: conv features -> final output
    classifier_input = tf.keras.Input(shape=last_conv_layer.output.shape[1:])
    x = classifier_input
    # skip index 0 because it is base_model in your Sequential
    for layer in model.layers[1:]:
        x = layer(x)
    classifier_model = tf.keras.models.Model(classifier_input, x)

    # gradient tape
    with tf.GradientTape() as tape:
        conv_output = last_conv_layer_model.predict(img_array)  # (1, h, w, c)
        tape.watch(conv_output)
        preds = classifier_model.predict(conv_output)           # (1, num_classes)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, conv_output)      # (1, h, w, c)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))   # (c,)

    conv_output = conv_output[0]                           # (h, w, c)
    heatmap = conv_output @ pooled_grads[..., tf.newaxis]  # (h, w, 1)
    heatmap = tf.squeeze(heatmap)                          # (h, w)

    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.math.reduce_max(heatmap) + tf.keras.backend.epsilon())
    return heatmap.numpy()

def save_and_display_gradcam(img_path, heatmap, cam_path, alpha=0.6):
    """
    Overlay heatmap on original image and save.
    """
    # original RGB image
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # resize heatmap
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

    superimposed_img = heatmap_color * alpha + img
    superimposed_img = np.clip(superimposed_img, 0, 255).astype("uint8")

    os.makedirs(os.path.dirname(cam_path), exist_ok=True)
    cv2.imwrite(cam_path, cv2.cvtColor(superimposed_img, cv2.COLOR_RGB2BGR))

    return img, superimposed_img

In [4]:
model_mobile = tf.keras.models.load_model("/workspaces/mobilnet-rice-leaf/rice_leaf_disease_mobilenet.keras")
model_mobile.trainable = False
last_conv_mobile = "conv_1"
base_mobile, _ = get_base_model(model_mobile)

2026-03-01 13:54:30.205188: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


ValueError: Layer "batch_normalization" expects 1 input(s), but it received 2 input tensors. Inputs received: [<KerasTensor shape=(None, 7, 7, 960), dtype=float32, sparse=False, ragged=False, name=keras_tensor_427>, <KerasTensor shape=(None, 7, 7, 960), dtype=float32, sparse=False, ragged=False, name=keras_tensor_428>]

In [38]:
from datetime import datetime

input_path = '/workspaces/mobilnet-rice-leaf/IMG_20190419_123649.jpg'

img_mobile = tf.keras.preprocessing.image.load_img(
    input_path, target_size=(config.IMG_SIZE_MOBILE, config.IMG_SIZE_MOBILE)
)
img_array_mobile = tf.keras.preprocessing.image.img_to_array(img_mobile)
img_array_mobile = np.expand_dims(img_array_mobile, axis=0)
img_array_mobile = preprocess_input(img_array_mobile)

preds_mobile = model_mobile.predict(img_array_mobile, verbose=0)[0]
pred_class_mobile = int(np.argmax(preds_mobile))
pred_conf_mobile = float(preds_mobile[pred_class_mobile])
class_label = ['bacterial_leaf_blight', 'brown_spot', 'leaf_blast', 'healthy_rice_leaf']
label = class_label[pred_class_mobile]

heatmap_mobile = make_gradcam_heatmap(
    img_array_mobile,
    model_mobile,
    last_conv_layer_name=last_conv_mobile,
    pred_index=pred_class_mobile
)

cam_path_mobile = f'{config.OUTPUT_DIR}/gradcam/{datetime.now()}_{label}.jpg'
_, gradcam_img_mobile = save_and_display_gradcam(input_path, heatmap_mobile, cam_path_mobile)


In [39]:
pred_class_mobile, pred_conf_mobile, class_label[pred_class_mobile]

(2, 0.42376908659935, 'leaf_blast')